In [ ]:
!pip install langgraph langchain-ollama langchain-community langchain-chroma pypdf

# Running Ollama in the background

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import os
import asyncio

async def run_ollama_serve():

    process = await asyncio.create_subprocess_shell(
        "ollama serve",
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    print("Ollama serve process started.")

    await process.wait()

asyncio.create_task(run_ollama_serve())

!sleep 5

In [ ]:
import os
import asyncio
import time

async def run_ollama_serve():

    process = await asyncio.create_subprocess_shell(
        "ollama serve",
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    print("Ollama serve process started.")

    await process.wait()

async def check_ollama_status():
    for _ in range(10): # Try for up to 50 seconds
        try:
            process = await asyncio.create_subprocess_shell(
                "ollama list",
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE
            )
            stdout, stderr = await process.communicate()
            if process.returncode == 0:
                print("Ollama server is running.")
                return True
            else:
                print("Ollama server not responding, waiting...")
                await asyncio.sleep(5) # Wait for 5 seconds before retrying
        except FileNotFoundError:
             print("Ollama command not found, waiting...")
             await asyncio.sleep(5) # Wait for 5 seconds before retrying
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            await asyncio.sleep(5)
    return False

async def start_and_pull():
    if not await check_ollama_status():
        print("Ollama not running, starting server...")
        asyncio.create_task(run_ollama_serve())
        if not await check_ollama_status():
            print("Failed to start Ollama server.")
            return

    print("Attempting to pull mistral model...")
    process = await asyncio.create_subprocess_shell(
        "ollama pull mistral",
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )
    stdout, stderr = await process.communicate()
    print(stdout.decode())
    if process.returncode != 0:
      print(stderr.decode())
      print("Failed to pull mistral model.")
      return

# Use the existing event loop instead of asyncio.run()
loop = asyncio.get_event_loop()
loop.create_task(start_and_pull())

# The main component


In [ ]:
from google.colab import files
import os
import sys
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, SystemMessage
from operator import add as add_messages
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool
import time

print("Please upload your PDF file.")
uploaded = files.upload()

if not uploaded:
    print("\nNo file uploaded. Please run the cell again.")
    sys.exit()

pdf_path = list(uploaded.keys())[0]
print(f"\nProcessing '{pdf_path}'...")

# Check if mistral model is available, wait if not
print("Checking for mistral model...")
for _ in range(60): # Wait for up to 5 minutes (60 * 5 seconds)
    try:
        result = os.popen('ollama list').read()
        if 'mistral' in result:
            print("Mistral model found.")
            break
        else:
            print("Mistral model not found, waiting...")
            time.sleep(5)
    except Exception as e:
        print(f"Error checking ollama list: {e}, waiting...")
        time.sleep(5)
else:
    print("Mistral model not found after waiting. Please run the previous cell to pull it and ensure Ollama is running.")
    sys.exit()


llm = ChatOllama(model="mistral", temperature=0)
embeddings = OllamaEmbeddings(model="mistral")

pdf_loader = PyPDFLoader(pdf_path)
pages = pdf_loader.load()
print(f"PDF loaded successfully with {len(pages)} pages.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
pages_split = text_splitter.split_documents(pages)

vectorstore = Chroma.from_documents(documents=pages_split, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("In-memory vector store created.")

@tool
def retriever_tool(query: str) -> str:
    """Searches and returns relevant information from the document."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in the document."
    return "\n---\n".join([doc.page_content for doc in docs])

tools = [retriever_tool]
llm_with_tools = llm.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def should_continue(state: AgentState):
    tool_calls = getattr(state['messages'][-1], 'tool_calls', None)
    return tool_calls is not None and len(tool_calls) > 0

system_prompt = f"""You are an intelligent AI assistant that answers questions based on the document named '{pdf_path}'.
You must always use the retriever tool to answer the user's question, even if you think you know the answer."""

tools_dict = {t.name: t for t in tools}

def call_llm(state: AgentState):
    messages = [
    SystemMessage(content=system_prompt),
    *state["messages"]
]

    message = llm_with_tools.invoke(messages)
    return {'messages': [message]}

def take_action(state: AgentState):
    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
        print(f"⚙️ Calling Tool: {t['name']} with query: {t['args'].get('query', '')}")
        result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
    return {'messages': results}

graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm", should_continue, {True: "retriever_agent", False: END})
graph.add_edge("retriever_agent", "llm")
rag_agent = graph.compile()
print("Agent is ready!")

while True:
    user_input = input("\nWhat is your question (or type 'exit'): ")
    if user_input.lower() in ['exit', 'quit']:
        break

    messages = [HumanMessage(content=user_input)]
    result = rag_agent.invoke({"messages": messages})

    print("\n=== ANSWER ===")
    print(result['messages'][-1].content)
    print("==============")